# LangGraph Basic

In [ ]:
from dotenv import load_dotenv
import os

load_dotenv(override=True)

os.getenv("OPENAI_API_KEY")

In [ ]:
from langchain.chat_models import init_chat_model

llm = init_chat_model(
    model = "gpt-5-nano",
    temperature = 0
)

## 2. State 선언

In [ ]:
from typing import TypedDict

class ChatState(TypedDict):
    message: str
    response: str
    


## 3. Node 선언

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

def agent_node(state:ChatState):
    """에이전트 노드는 도구(TOOL)에 전달하기 전에 사용자의 쿼리를 재구성합니다."""
    prompt = ChatPromptTemplate.from_template(
        """사용자의 질문: {question}
        날씨를 검색하기 위한 도시 이름만 알려줘"""
    )
    chain = prompt | llm 
    output = chain.invoke({"question": state["message"]}).content
    return {"message":output, "response":""}

- 테스트

In [ ]:
test_state = {
    "message":"서울의 날씨를 알려줘?",
    "response":""
}

result = agent_node(test_state)

result

In [ ]:
def weather_node(state:ChatState):
    city = state["message"]
    response = f"{city}의 날씨는 맑습니다."
    return {"message":state["message"],"response":response}

In [ ]:
test_state = {
    "message": "Seoul",
    "response" : ""
}
result = weather_node(test_state)
result

## 그래프 구성

In [ ]:
from langgraph.graph import StateGraph, START, END

workflow = StateGraph(ChatState)
workflow.add_node("agent",agent_node)
workflow.add_node("weather",weather_node)

workflow.add_edge("agent","weather")
workflow.add_edge(START, "agent")
workflow.add_edge("weather",END)
graph = workflow.compile()

In [ ]:
from IPython.display import Image, display

png_bytes = graph.get_graph().draw_mermaid_png()

display(Image(data=png_bytes))

## 실행

In [ ]:
inputs = {
    "message": "서울의 날씨를 알려줘?",
    "response": ""
}

In [ ]:
response = graph.invoke(inputs)
response

## Conditional Edge

In [ ]:
def route_condition(state:ChatState) -> ChatState:
    city = state["message"]
    if city == "Seoul" or city == "서울":
        return "weather"
    else:
        return "end"

In [ ]:
workflow = StateGraph(ChatState)
workflow.add_node("agent",agent_node)
workflow.add_node("weather",weather_node)

#workflow.add_edge("agent","weather") 주석 처리 꼭 하기 !!
workflow.add_edge(START, "agent")
workflow.add_conditional_edges(
    "agent",
    route_condition,
    {
        "weather" : "weather",
        "end": END
    } 
)
workflow.add_edge("weather",END)
graph = workflow.compile()

In [ ]:
png_bytes = graph.get_graph().draw_mermaid_png()

display(Image(data=png_bytes))

In [ ]:
inputs = {
    "message": "부산의 날씨를 알려줘?",
    "response": ""
}

response = graph.invoke(inputs)
response

## 환율 그래프 예제

In [ ]:
# --- Node 1: Agent ---
def agent_node(state: ChatState) -> ChatState:
    """
    Agent checks if it's a currency conversion query.
    If yes, extracts amount/from_cur/to_cur as plain text.
    """
    prompt = ChatPromptTemplate.from_template("""
    The user said: "{question}".

    - If it's a currency conversion, extract:
      - amount (number)
      - from currency (3-letter code)
      - to currency (3-letter code)

    Reply in this format ONLY:
    amount=<number>, from=<FROM>, to=<TO>

    - - If it is NOT a currency conversion query, you MUST reply exactly with this phrase:
    no_tool=This is not a currency conversion query.
    """)
    chain = prompt | llm
    # strip 양쪽끝의 공백을 없앰
    output = chain.invoke({"question": state["message"]}).content.strip() 

    if output.startswith("amount="):
        parts = dict(item.split("=") for item in output.split(", "))
        return {
            "message": state["message"],
            "response": "USE_TOOL",
            "amount": float(parts["amount"]),
            "from_cur": parts["from"],
            "to_cur": parts["to"],
        }
    else:
        return {"message": state["message"], "response": output.replace("no_tool=", "").strip()}

In [ ]:
from typing import TypedDict

class ChatState(TypedDict):
    message: str
    response: str

- 테스트

In [ ]:
test_state = {
    "message": "What's the price of 10 USD to KRW?"
}
agent_node(test_state)

- API 환율 정보 노드

In [ ]:
import requests

from_cur = 'USD'
to_cur = 'KRW'
amount = 1
url = f"https://api.frankfurter.app/latest?amount={amount}&from={from_cur}&to={to_cur}"
response = requests.get(url)
response

In [ ]:
response.json()

In [ ]:
# --- Node 2: ConvertCurrency tool ---
def convert_currency(state: ChatState) -> ChatState:
    """Dynamic currency conversion using Frankfurter API."""
    amount = state.get("amount", 1.0)
    from_cur = state.get("from_cur", 'USD')
    to_cur = state.get("to_cur", 'KRW')

    url = f"https://api.frankfurter.app/latest?amount={amount}&from={from_cur}&to={to_cur}"
    response = requests.get(url)

    if response.status_code == 200:
        data = response.json()
        rate = list(data["rates"].values())[0]
        raw_response = f"{amount} {from_cur} = {rate:.2f} {to_cur}"
    else:
        raw_response = "Error fetching conversion rate."

    return {"message": state["message"], "response": raw_response}

- 테스트

In [ ]:
test_state = {
    'message': "What's the price of 10 USD to KRW?",
    'response': 'USE_TOOL',
    'amount': 10.0,
    'from_cur': 'USD',
    'to_cur': 'KRW'
}

convert_currency(test_state)

In [ ]:
def response_agent_node(state: ChatState) -> ChatState:
    """Refines the response from currency_tool into human-readable text."""
    prompt_text = f"""
    Here is a currency conversion result: '{state['response']}'.
    Please convert it into a clear, human-readable sentence.
    """
    refined_response = llm.invoke(prompt_text).content.strip()
    return {"message": state["message"], "response": refined_response}

In [ ]:
test_state = {
    'message': "What's the price of 10 USD to KRW?",
    'response': '10.0 USD = 14676.00 KRW'
}

response_agent_node(test_state)

## 그래프 구성

In [ ]:
# --- Build the Graph ---
workflow = StateGraph(ChatState)

# Add nodes
workflow.add_node("agent", agent_node)
workflow.add_node("convert_currency_tool", convert_currency)
workflow.add_node("response_agent", response_agent_node)

In [ ]:
from langgraph.graph import StateGraph, END

# Conditional routing function
def route_condition(state: ChatState) -> ChatState:
    if state["response"] == "USE_TOOL":
        return "convert_currency_tool"
    else:
        return END   # <-- use END constant, not "__end__"

# Conditional edges (agent can go either to currency_tool or END)
workflow.add_conditional_edges(
    "agent",
    route_condition,
    {
        "convert_currency_tool": "convert_currency_tool",
        END: END,
    }
)


# Normal edges
workflow.add_edge("convert_currency_tool", "response_agent")
workflow.add_edge("response_agent", END)

# Set entry point
workflow.set_entry_point("agent")

# Compile
graph = workflow.compile()

In [ ]:
from IPython.display import display, Markdown

# Get the Mermaid code
mermaid_code = graph.get_graph().draw_mermaid()

# Display in Jupyter
display(Markdown(f"```mermaid\n{mermaid_code}\n```"))

In [ ]:
graph

In [ ]:
inputs = {"message":"What's the price of 10 USD to JPY?"}
graph.invoke(inputs)

In [ ]:
inputs = {"message":"Hey, Whats the capital of Korea?"}
graph.invoke(inputs)